# Smart IDS v3 — Notebook 04: Unsupervised Zero-Day Anomaly Detection
Autoencoder Architecture, Benign-Only Training, and Anomaly Threshold Calibration

---

### What You Will Learn
1. **The Closed-Set Dilemma**: Why supervised classifiers fail against new, unseen cyber threats.
2. **Zero-Day Attacks**: Defining zero-day vulnerabilities in security operations.
3. **ML Analogy**: Using an Undercomplete Autoencoder as an anomaly detector.
4. **Benign-Only Training**: Training solely on normal traffic under a Mean Squared Error (MSE) objective.
5. **Threshold Calibration**: Calibrating the decision boundary at the 95th percentile of benign validation reconstruction error.
6. **Binary Evaluation**: Assessing Anomaly vs Benign performance on held-out test data.

---

### Prerequisites
- Completion of Notebooks 01 through 03.
- Basic understanding of autoencoders (Encoder, Bottleneck Latent Space, Decoder).


## 1. Security Concept: The Zero-Day Problem & Closed-Set Supervised Learning

> **Security Concept: What is a Zero-Day Threat?**
> A **zero-day** is a vulnerability or exploit that has never been seen before. There are zero days of warning, zero public signatures, and zero labeled examples in training datasets.
> 
> *The Closed-Set Assumption in Supervised Learning:*
> Supervised models (like MLP, CNN, and XGBoost) can only classify attacks they were explicitly trained to recognize. If a completely new attack arrives, a supervised model will be forced to guess one of its existing labels (or mistake it for Benign).
> 
> *The Solution: Anomaly Detection via Autoencoder:*
> Instead of learning what attacks look like, we train an Autoencoder to learn what **BENIGN (normal) traffic** looks like.
> When normal traffic passes through, the Autoencoder reconstructs it with low error.
> When an abnormal zero-day attack arrives, the Autoencoder fails to reconstruct it, producing a **high reconstruction error**!


In [1]:
import sys
from pathlib import Path

cur = Path.cwd()
root = cur if (cur / 'src').exists() else cur.parent
if str(root / 'src') not in sys.path:
    sys.path.insert(0, str(root / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from smart_ids.config import (
    CLASS_NAMES,
    BENIGN_CLASS_NAME,
    MODELS_DIR,
    PROCESSED_DATA_DIR,
    SEED,
)
from smart_ids.models import build_autoencoder
from smart_ids.training import train_autoencoder, get_callbacks
from smart_ids.evaluation import (
    compute_reconstruction_error,
    calibrate_anomaly_threshold,
    evaluate_autoencoder_binary,
)
from smart_ids.artifacts import save_anomaly_threshold

np.random.seed(SEED)
tf.random.set_seed(SEED)
print(f"Autoencoder environment initialized with seed {SEED}.")


Autoencoder environment initialized with seed 42.


## 2. Isolating Benign Traffic for Self-Supervised Training

**What we do and why:**
We filter our scaled training and validation datasets to keep **ONLY samples where Label == 'BENIGN'** (class index 0). All attacks are strictly excluded from the Autoencoder's training!


In [2]:
# Load scaled splits from Notebook 02
X_train_scaled = np.load(PROCESSED_DATA_DIR / 'X_train_scaled.npy')
y_train = np.load(PROCESSED_DATA_DIR / 'y_train.npy')
X_val_scaled = np.load(PROCESSED_DATA_DIR / 'X_val_scaled.npy')
y_val = np.load(PROCESSED_DATA_DIR / 'y_val.npy')
X_test_scaled = np.load(PROCESSED_DATA_DIR / 'X_test_scaled.npy')
y_test = np.load(PROCESSED_DATA_DIR / 'y_test.npy')

# Filter for Benign only (Class Index 0)
benign_idx = 0
train_benign_mask = (y_train == benign_idx)
val_benign_mask = (y_val == benign_idx)

X_benign_train = X_train_scaled[train_benign_mask]
X_benign_val = X_val_scaled[val_benign_mask]

n_features = X_benign_train.shape[1]
print(f"Benign Training Samples:   {X_benign_train.shape[0]:,}")
print(f"Benign Validation Samples: {X_benign_val.shape[0]:,}")


Benign Training Samples:   15,026
Benign Validation Samples: 2,652


## 3. Building and Training the Undercomplete Autoencoder

**What we do and why:**
We construct the Autoencoder:
- **Encoder**: `Dense(256) -> BN -> ReLU -> Dense(128) -> BN -> ReLU -> Dense(64) -> Latent(32)`.
- **Decoder**: `Dense(64) -> BN -> ReLU -> Dense(128) -> BN -> ReLU -> Dense(256) -> Dense(n_features, linear)`.
Linear activation is essential on the reconstruction layer because features were standardized to real-valued $z$-scores ($\mu=0, \sigma=1$).


In [3]:
autoencoder, encoder, decoder = build_autoencoder(
    input_dim=n_features,
    latent_dim=32,
    learning_rate=5e-4,
)

ae_callbacks = get_callbacks(
    "autoencoder",
    patience_es=10,
    checkpoint_name="best_autoencoder.keras",
)

autoencoder_trained, ae_history = train_autoencoder(
    autoencoder,
    X_benign_train,
    X_benign_val,
    epochs=15,
    batch_size=1024,
    callbacks=ae_callbacks,
)
print("Autoencoder training complete. Saved to models/best_autoencoder.keras.")


Epoch 1/15


 1/15 ━━━━━━━━━━━━━━━━━━━━ 33s 2s/step - loss: 2.4487

 8/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 2.1421

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.9308

15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - loss: 1.9308 - val_loss: 0.9502 - learning_rate: 5.0000e-04


Epoch 2/15


... (66 outputs left out)

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.7605

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.7605 - val_loss: 0.8057 - learning_rate: 5.0000e-04


Restoring model weights from the end of the best epoch: 15.


Autoencoder training complete. Saved to models/best_autoencoder.keras.


## 4. Calibrating the Anomaly Detection Threshold

**What we do and why:**
We compute the reconstruction error (Mean Squared Error) for each flow in `X_benign_val`. 
We set `ANOMALY_THRESHOLD` to the **95th percentile** of these benign errors.
This mathematically guarantees that the expected false-positive rate on known benign traffic is bounded at ~5%.


In [4]:
# Calibrate 95th percentile threshold on unseen benign validation flows
threshold = calibrate_anomaly_threshold(autoencoder_trained, X_benign_val, percentile=95.0)

# Save threshold metadata to models/anomaly_threshold.json
save_anomaly_threshold(threshold, percentile=95.0)
print(f"Calibrated Anomaly Decision Threshold: {threshold:.6f}")


Calibrated Anomaly Decision Threshold: 1.620818


## 5. Visualizing Reconstruction Error Distributions

Let's compare the reconstruction error distributions of **Benign Test Flows** vs **Attack Test Flows**.


In [5]:
test_errors = compute_reconstruction_error(autoencoder_trained, X_test_scaled)
is_attack_test = (y_test != benign_idx)

benign_test_errors = test_errors[~is_attack_test]
attack_test_errors = test_errors[is_attack_test]

plt.figure(figsize=(10, 5))
plt.hist(benign_test_errors, bins=50, range=(0, threshold * 3), alpha=0.6, label='Benign Test Flows', color='#10b981', density=True)
plt.hist(attack_test_errors, bins=50, range=(0, threshold * 3), alpha=0.6, label='Attack Test Flows', color='#ef4444', density=True)
plt.axvline(threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold (95th %ile) = {threshold:.4f}')

plt.title("Reconstruction Error Distribution: Benign vs Attacks", fontsize=13, fontweight='bold')
plt.xlabel("Reconstruction Error (MSE)", fontsize=11)
plt.ylabel("Density", fontsize=11)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


<Figure size 1000x500 with 1 Axes>

## 6. Binary Evaluation on Held-Out Test Data

**What we do and why:**
Any flow with reconstruction error > `ANOMALY_THRESHOLD` is predicted as `1` (Anomaly).
We evaluate binary Accuracy, Precision, Recall, and F1 score against ground truth `(y_test != BENIGN)`.


In [6]:
y_pred_binary = (test_errors > threshold).astype(int)
y_true_binary = is_attack_test.astype(int)

ae_metrics = evaluate_autoencoder_binary(y_true_binary, y_pred_binary)
print("--- Autoencoder Binary Anomaly Evaluation ---")
for k, v in ae_metrics.items():
    print(f"  {k}: {v if isinstance(v, str) else round(v, 4)}")


--- Autoencoder Binary Anomaly Evaluation ---
  Accuracy: 0.8349
  Precision: 0.5201
  Recall: 0.2254
  F1-Score: 0.3145
  Latency (us): 0.0
  Evaluation Type: Binary (Zero-Day Anomaly vs Benign)


---
## Summary and Next Notebook

In this notebook:
- We trained an **Undercomplete Autoencoder** exclusively on benign network traffic.
- We calibrated a **95th percentile reconstruction threshold** on unseen benign validation data.
- We confirmed that malicious traffic exhibits substantially higher reconstruction error.
- We evaluated the Autoencoder under a **binary framing** (Normal vs Anomaly).

**Next up:** In **Notebook 05: Evaluation and Comparison**, we will benchmark per-sample inference latency for all 4 models, compare their performance side-by-side, generate confusion matrices, and plot multi-class One-vs-Rest ROC curves!
